In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

marketing = spark.table("lh_silver_game.marketing_spend_clean")
players = spark.table("lh_silver_game.players_clean")
purchases = spark.table("lh_silver_game.purchases_clean")

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 3, Finished, Available, Finished, False)

In [2]:
creative_marketing = (
    marketing
    .groupBy(
        "creative_id",
        "creative_type",
        "creative_concept"
    )
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks"),
        F.sum("installs").alias("installs"),
        F.sum("spend_usd").alias("spend_usd")
    )
    .withColumn(
        "ctr",
        F.col("clicks") / F.col("impressions")
    )
    .withColumn(
        "cvr",
        F.col("installs") / F.col("clicks")
    )
    .withColumn(
        "cpi",
        F.col("spend_usd") / F.col("installs")
    )
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 4, Finished, Available, Finished, False)

In [3]:
purchase_by_player = (
    purchases
    .groupBy("player_id")
    .agg(
        F.sum("price_usd").alias("purchase_revenue")
    )
)

creative_player_quality = (
    players
    .filter(F.col("creative_id").isNotNull())
    .select(
        "player_id",
        "creative_id"
    )
    .join(
        purchase_by_player,
        on="player_id",
        how="left"
    )
    .fillna({
        "purchase_revenue": 0.0
    })
    .withColumn(
        "is_payer",
        F.when(
            F.col("purchase_revenue") > 0,
            1
        ).otherwise(0)
    )
    .groupBy("creative_id")
    .agg(
        F.count("*").alias("attributed_players"),
        F.sum("is_payer").alias("payers"),
        F.sum("purchase_revenue").alias("purchase_revenue")
    )
    .withColumn(
        "payer_conversion",
        F.col("payers") / F.col("attributed_players")
    )
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 5, Finished, Available, Finished, False)

In [4]:
creative_performance = (
    creative_marketing
    .join(
        creative_player_quality,
        on="creative_id",
        how="left"
    )
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 6, Finished, Available, Finished, False)

In [5]:
print("Creative count:", creative_performance.count())

display(
    creative_performance
    .orderBy("creative_id")
    .limit(20)
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 7, Finished, Available, Finished, False)

Creative count: 69


SynapseWidget(Synapse.DataFrame, 054e749b-1019-46a7-bfe2-30438a4c3bc5)

In [6]:
creative_performance = (
    creative_performance
    .withColumn(
        "roas",
        F.when(
            F.col("spend_usd") > 0,
            F.col("purchase_revenue") / F.col("spend_usd")
        )
    )
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 8, Finished, Available, Finished, False)

In [7]:
creative_performance = (
    creative_performance
    .withColumn(
        "revenue_per_attributed_player",
        F.when(
            F.col("attributed_players") > 0,
            F.col("purchase_revenue") / F.col("attributed_players")
        )
    )
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 9, Finished, Available, Finished, False)

In [8]:
display(
    creative_performance
    .select(
        "creative_id",
        "creative_type",
        "creative_concept",
        "impressions",
        "clicks",
        "installs",
        "ctr",
        "cvr",
        "cpi",
        "attributed_players",
        "payers",
        "payer_conversion",
        "purchase_revenue",
        "roas",
        "revenue_per_attributed_player"
    )
    .orderBy("creative_id")
    .limit(20)
)

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e196a304-0522-4c03-886d-61255d3465ec)

In [9]:
creative_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.creative_performance")

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 11, Finished, Available, Finished, False)

In [10]:
df_check = spark.table("lh_gold_game.creative_performance")

print("Saved row count:", df_check.count())
display(df_check.limit(10))

StatementMeta(, a5a28d82-4ddb-4679-ab84-2e127283a194, 12, Finished, Available, Finished, False)

Saved row count: 69


SynapseWidget(Synapse.DataFrame, 00916dfa-be9b-402f-b3e6-1407c76c54b0)